<a href="https://colab.research.google.com/github/embark-cybertraining/embark-scratch-notebooks/blob/main/obis_filter_by_areaid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Query OBIS for one species in one marine region and display summary information and map locations.

This notebook was constructed with the goal of porting functionality from an r script using "robis" OBIS_filter_by_areaid.R to python using "pyobis".

Strategy was revised to do an obis api call directly instead of through the pythong client pyobis in order to use the areaid param instead of passing an area geometry which would be required to use pyobis.

OBIS API https://api.obis.org/
pyobis https://iobis.github.io/pyobis/

ChatGPT5.3 was utilized to help port the functionality with oversight, editing for purpose, and review by the embark project participants.

In [24]:
"""
Query OBIS for one species in one marine region and display summary information.

Purpose:
- Retrieve OBIS occurrence records for a scientific name within an areaid
- Display summary information
- Map locations of the occurrence records for data exploration

Usage:
- Edit SCIENTIFIC_NAME AREA_NAME, and AREA_ID as needed

"""

import requests
import folium
from datetime import datetime
import pandas as pd
from IPython.display import display

SCIENTIFIC_NAME = "Pyrosoma atlanticum"
AREA_ID = 40003
AREA_NAME = 'California Current'
RECORD_LIMIT = 10000 #query up to this many records

#Query up to 10k records in one call
def fetch_obis_occurrences(scientific_name, area_id, size):
    r = requests.get(
        "https://api.obis.org/v3/occurrence",
        params={
            "scientificname": scientific_name,
            "areaid": area_id,
            "size": size,
        },
        timeout=60,
    )
    r.raise_for_status() #raises an exception if the request failed
    return pd.DataFrame(r.json().get("results", []))

records = fetch_obis_occurrences(SCIENTIFIC_NAME, AREA_ID, RECORD_LIMIT)

## show summary info
print(f"Scientific name: {SCIENTIFIC_NAME}")
print(f"Area ID: {AREA_ID}")
print(f"Total records retrieved: {len(records)}")
if len(records) <= RECORD_LIMIT:
    print("All records retrieved.")

Scientific name: Pyrosoma atlanticum
Area ID: 40003
Total records retrieved: 483
All records retrieved.


In [4]:
# If you want to save as a csv file:

output_file = "obis_occurrences_pyrosoma_atlanticum_california_current.csv"
records.to_csv(output_file, index=False)
print(f"Saved to: {output_file}")

Saved to: obis_occurrences_pyrosoma_atlanticum_california_current.csv


In [18]:
#More summary info

if records.empty:
    print("No records returned.")
else:
    print("\nColumns returned:")
    print(sorted(records.columns.tolist()))

    # Optional preview of key fields
    preview_cols = [
        "scientificName",
        "decimalLatitude",
        "decimalLongitude",
        "eventDate",
        "minimumDepthInMeters",
        "maximumDepthInMeters",
        "occurrenceStatus",
        "datasetID",
        "occurrenceID"
    ]

    print("\nPreview of key fields (random 10 lines)")
    display(records[preview_cols].sample(10))

    # Unique aphiaID values with WoRMS links

    aphia_values = pd.Series(records["aphiaID"].dropna().unique(), name="aphiaID")
    aphia_df = pd.DataFrame({"aphiaID": aphia_values})
    aphia_df["WoRMS_link"] = aphia_df["aphiaID"].apply(
        lambda x: f"https://marinespecies.org/aphia.php?p=taxdetails&id={int(x)}"
        if pd.notna(x) else None
    )

    display(aphia_df.sort_values("aphiaID").reset_index(drop=True))

    # Unique datasets, add OBIS links

    records["datasetURL"] = 'https://obis.org/dataset/'+ records['dataset_id'].astype(str)

    # Extract unique combinations
    datasets = (
        records[['dataset_id',"datasetName",'datasetID','datasetURL']]
        .drop_duplicates()
        .sort_values(by="datasetName")
        .reset_index(drop=True)
    )

    print(f"\nTotal unique datasets: {len(datasets)}")
    display(datasets)




Columns returned:
['absence', 'aphiaID', 'basisOfRecord', 'bathymetry', 'class', 'classid', 'country', 'datasetID', 'datasetName', 'dataset_id', 'date_end', 'date_mid', 'date_start', 'date_year', 'decimalLatitude', 'decimalLongitude', 'depth', 'dropped', 'eventDate', 'eventID', 'eventRemarks', 'eventTime', 'family', 'familyid', 'flags', 'genus', 'genusid', 'geodeticDatum', 'higherClassification', 'id', 'identificationRemarks', 'individualCount', 'institutionID', 'kingdom', 'kingdomid', 'language', 'license', 'lifeStage', 'locality', 'locationID', 'locationRemarks', 'marine', 'materialSampleID', 'maximumDepthInMeters', 'maximumElevationInMeters', 'minimumDepthInMeters', 'minimumElevationInMeters', 'modified', 'node_id', 'occurrenceID', 'occurrenceStatus', 'order', 'orderid', 'organismQuantity', 'organismQuantityType', 'originalScientificName', 'ownerInstitutionCode', 'parentEventID', 'phylum', 'phylumid', 'recordedBy', 'references', 'sampleSizeUnit', 'sampleSizeValue', 'samplingEffort'

,scientificName,decimalLatitude,decimalLongitude,eventDate,minimumDepthInMeters,maximumDepthInMeters,occurrenceStatus,datasetID,occurrenceID
2,Pyrosoma atlanticum,35.705200,-121.507200,2014-05-30T07:29:06Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,98fecfcb-a925-43d2-9215-50881485c7c8
1,Pyrosoma atlanticum,36.744500,-121.977000,2012-05-31T07:49:21Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,eb38cddf-cc12-4bdf-9388-3629930b20e2
8,Pyrosoma atlanticum,36.979300,-122.289200,2012-06-11T04:40:56Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,6becb1ba-3107-498f-a660-7cb836f9f8ec
5,Pyrosoma atlanticum,33.379800,-119.773300,2013-06-09T04:01:14Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,8dfbb85e-5670-437d-b3f8-864ae46a7c31
9,Pyrosoma atlanticum,36.983200,-122.427200,2013-05-28T07:53:35Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,1dcb4223-1723-4301-b1ac-0d5dcb7630fc
6,Pyrosoma atlanticum,35.000700,-120.888800,2012-05-29T06:00:20Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,3be360a4-f0b0-451c-8edc-a03d012ecb4d
0,Pyrosoma atlanticum,41.494953,-125.095577,2023-04-20T22:07:39Z,2316.368896,2316.368896,present,noaa-oer-okeanos-ex2301,noaa-oer-okeanos-ex2301:OKEX2301_ROV-D2_18S_na...
3,Pyrosoma atlanticum,35.709500,-121.507800,2012-05-30T07:28:18Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,b0be9481-183e-437a-a178-1110c50d3e14
4,Pyrosoma atlanticum,40.500500,-124.617700,2014-06-14T06:16:14Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,604fa7ea-138e-4b9f-ae31-e00802f6a40c
7,Pyrosoma atlanticum,36.300700,-122.012700,2014-06-01T05:31:09Z,25.000000,30.000000,present,RockfishRecruitmentAndEcosystemAssessmentSurve...,2e152bbe-2b3a-4092-b58c-9b93956b822b


,aphiaID,WoRMS_link
0,137250,https://marinespecies.org/aphia.php?p=taxdetai...



Total unique datasets: 2


,dataset_id,datasetName,datasetID,datasetURL
0,5b6251f6-a7a5-4dc9-994d-c9504b54776f,Rockfish Recruitment and Ecosystem Assessment ...,RockfishRecruitmentAndEcosystemAssessmentSurve...,https://obis.org/dataset/5b6251f6-a7a5-4dc9-99...
1,71c2c816-7e94-40b9-8e28-8172d9c5fefb,NaN,noaa-oer-okeanos-ex2301,https://obis.org/dataset/71c2c816-7e94-40b9-8e...


In [6]:
# Plot OBIS occurrence records from the `records` DataFrame

def plot_obis_records(records_df):
    """
    Return a folium map of OBIS occurrence points.
    """
    required = {"decimalLatitude", "decimalLongitude"}
    missing = required - set(records_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = records_df.dropna(subset=list(required))
    if df.empty:
        raise ValueError("No valid coordinate rows.")

    m = folium.Map(
        location=[df["decimalLatitude"].mean(), df["decimalLongitude"].mean()],
        zoom_start=4
    )

    #add this info in a popup using html table
    preview_cols = ["scientificName","decimalLatitude","decimalLongitude",
                    "eventDate","minimumDepthInMeters","maximumDepthInMeters","occurrenceID" ]

    cols = [c for c in preview_cols if c in df.columns]

    for _, r in df.iterrows():
        rows = [
            f"<tr><th>{c}</th><td>{r[c]}</td></tr>"
            for c in cols if pd.notna(r.get(c))
        ]

        popup = folium.Popup(f"<table>{''.join(rows)}</table>", max_width=300)

        folium.CircleMarker(
            location=[r.decimalLatitude, r.decimalLongitude],
            radius=3,
            weight=1,
            fill=True,
            popup=popup,
        ).add_to(m)

    return m

obis_map = plot_obis_records(records)
obis_map

# Source Datasets and Metadata

Dataset info pattern
https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f
* You can see "Overview" and "Data Quality" sections which include information about quality flags, missing data identifiers, coordinate precision, etc.

You can get the unique set of dataset_id column in records to look at metadata for any information you need about the source datasets in obis.

In [7]:
# We alread got the unique list of datasets from records
datasets

,dataset_id,datasetName,datasetID,datasetURL
0,5b6251f6-a7a5-4dc9-994d-c9504b54776f,Rockfish Recruitment and Ecosystem Assessment ...,RockfishRecruitmentAndEcosystemAssessmentSurve...,https://obis.org/dataset/5b6251f6-a7a5-4dc9-99...
1,71c2c816-7e94-40b9-8e28-8172d9c5fefb,NaN,noaa-oer-okeanos-ex2301,https://obis.org/dataset/71c2c816-7e94-40b9-8e...
2,c5687a17-e454-40f9-9a4b-d04b2c812d74,NaN,NaN,https://obis.org/dataset/c5687a17-e454-40f9-9a...
3,aa16d305-d413-4c4a-90be-b1ec3298d58d,NaN,NaN,https://obis.org/dataset/aa16d305-d413-4c4a-90...


In [8]:
#You can visit the links for more info about the dataset
datasets["datasetURL"].dropna().tolist()


['https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f',
 'https://obis.org/dataset/71c2c816-7e94-40b9-8e28-8172d9c5fefb',
 'https://obis.org/dataset/c5687a17-e454-40f9-9a4b-d04b2c812d74',
 'https://obis.org/dataset/aa16d305-d413-4c4a-90be-b1ec3298d58d']

In [9]:
# Dataset citation for OBIS
# * Citation based on https://manual.obis.org/citing.html

#citaion builder based on this script's variables
now = datetime.now()
year = now.year
access_date = now.strftime("%B %d, %Y")

citation = (
    f"Ocean Biodiversity Information System (OBIS). ({year}). "
    f"Occurrence records of {SCIENTIFIC_NAME} in the {AREA_NAME} (LME {AREA_ID}). "
    f"https://obis.org (accessed {access_date})."
)

print("OBIS citation:\n")
print(citation)

OBIS citation:

Ocean Biodiversity Information System (OBIS). (2026). Occurrence records of Pyrosoma atlanticum in the California Current (LME 40003). https://obis.org (accessed May 08, 2026).


# Testing same functionality but with pyobis call with a geometry string (not an area id)



In [25]:
!pip install pyobis



  Using cached pyobis-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached requests_cache-1.3.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached cattrs-26.1.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached url_normalize-3.0.0-py3-none-any.whl.metadata (7.5 kB)
Using cached pyobis-1.6.1-py3-none-any.whl (29 kB)
Using cached requests_cache-1.3.1-py3-none-any.whl (70 kB)
Using cached cattrs-26.1.0-py3-none-any.whl (73 kB)
Using cached url_normalize-3.0.0-py3-none-any.whl (16 kB)


In [34]:
#investigating this region to see the info about it before using the geometry string

import requests
import pandas as pd

MRGID = 8549  # California Current (Marine Regions)

url = f"https://www.marineregions.org/rest/getGazetteerRecordByMRGID.json/{MRGID}/"
r = requests.get(url, timeout=60)
r.raise_for_status()

data = r.json()

# Flatten key fields for easy viewing
info = {
    "MRGID": data.get("MRGID"),
    "preferred_name": data.get("preferredGazetteerName"),
    "place_type": data.get("placeType"),
    "status": data.get("status"),
    "latitude": data.get("latitude"),
    "longitude": data.get("longitude"),
}

df = pd.DataFrame([info])
print(df)

# Full raw record if needed
data

   MRGID      preferred_name              place_type    status  latitude  \
0   8549  California Current  Large Marine Ecosystem  altclass  33.25191   

   longitude  
0 -122.51089  


{'MRGID': 8549,
 'gazetteerSource': 'Large Marine Ecosystems of the World',
 'placeType': 'Large Marine Ecosystem',
 'latitude': 33.25191,
 'longitude': -122.51089,
 'minLatitude': 21.83586,
 'minLongitude': -132.11243,
 'maxLatitude': 47.90313,
 'maxLongitude': -108.75468,
 'precision': 1784230,
 'preferredGazetteerName': 'California Current',
 'preferredGazetteerNameLang': 'English',
 'status': 'altclass',
 'accepted': 8549}

In [61]:
#get the geometry

import re
import requests

MRGID = 8549  # California Current (Marine Regions)

url = f"https://www.marineregions.org/rest/getGazetteerGeometries.jsonld/{MRGID}/"
r = requests.get(url, timeout=60)
r.raise_for_status()

data = r.json()
geometry_wkt = None

for geom in data.get("mr:hasGeometry", []):
    wkt = geom.get("gsp:asWKT")
    if wkt:
        geometry_wkt = re.sub(r"<[^>]+>\s*", "", wkt).strip()
        break

print("Preview the geom")
print(geometry_wkt[0:100])
print("Geometry length="+str(len(geometry_wkt)))

Preview the geom
POLYGON ((-109.91147613 22.8783226, -108.75467682 21.83747101, -108.7578125 21.83586121, -109.616462
Geometry length=108685


In [55]:
#look at a query using example species and geom pybix occurences examples use too to make sure it works
from pyobis import occurrences
import pandas as pd

query = occurrences.search(
    scientificname="Mola mola",
    geometry="POLYGON((30.1 10.1, 10 20, 20 40, 40 40, 30.1 10.1))",
    size=RECORD_LIMIT,
    cache=True
)

query.execute()

df = pd.DataFrame(query.data["results"])

display(df.head())

print(query.api_url)
print(query.mapper_url)

,associatedReferences,basisOfRecord,bibliographicCitation,brackish,catalogNumber,class,classid,collectionCode,coordinatePrecision,datasetID,...,absence,originalScientificName,aphiaID,category,flags,bathymetry,shoredistance,sst,sss,coordinateUncertaintyInMeters
0,"[{""crossref"":{""citeinfo"":{""origin"":""Halpin, P....",HumanObservation,"[{""crossref"":{""citeinfo"":{""origin"":""Halpin, P....",False,2322_22,Teleostei,293496,2322,1e-06,2322,...,False,Mola mola,127405,VU,[NO_DEPTH],4338,32472,20.85,38.22,NaN
1,"[{""crossref"":{""citeinfo"":{""origin"":""Halpin, P....",HumanObservation,"[{""crossref"":{""citeinfo"":{""origin"":""Halpin, P....",False,1158_2698,Teleostei,293496,1158,1e-05,1158,...,False,Mola mola,127405,VU,[NO_DEPTH],248,9219,20.44,38.52,1.11


https://api.obis.org/v3/occurrence?scientificname=Mola+mola&geometry=POLYGON%28%2830.1+10.1%2C+10+20%2C+20+40%2C+40+40%2C+30.1+10.1%29%29&offset=0&mof=False&size=10000
https://mapper.obis.org/?taxonid=127405&scientificname=Mola+mola&geometry=POLYGON%28%2830.1+10.1%2C+10+20%2C+20+40%2C+40+40%2C+30.1+10.1%29%29&offset=0&mof=False&size=10000


In [56]:
#look at a query using example species and geom pybix occurences examples use too. May not be able to get around the url too large calling this way.
from pyobis import occurrences
import pandas as pd

query = occurrences.search(
    scientificname=SCIENTIFIC_NAME,
    geometry=geometry_wkt,
    size=RECORD_LIMIT,
    cache=True
)

query.execute()

df = pd.DataFrame(query.data["results"])

display(df.head())

print(query.api_url)
print(query.mapper_url)

HTTPError: 414 Client Error: Request-URI Too Large for url: https://api.obis.org/v3/occurrence?taxonid=137250&scientificname=Pyrosoma+atlanticum&geometry=POLYGON+%28%28-109.91147613+22.8783226%2C+-108.75467682+21.83747101%2C+-108.7578125+21.83586121%2C+-109.61646271+21.89846611%2C+-110.6988678+21.94595146%2C+-112.19152832+21.9803009%2C+-113.03099823+21.98961067%2C+-113.32858276+21.94040298%2C+-114.03866577+22.0181179%2C+-114.9164505+22.07999802%2C+-115.5743103+22.0719471%2C+-115.68096161+22.07064247%2C+-116.4107666+22.18305588%2C+-117.51317596+22.26417541%2C+-118.03858185+22.39151382%2C+-118.62602234+22.74551201%2C+-119.15538025+22.99394035%2C+-120.02405548+23.33173561%2C+-120.55944061+23.73477936%2C+-120.78397369+24.32380676%2C+-121.20969391+24.74063683%2C+-121.65570068+25.17365837%2C+-122.35441589+25.81252289%2C+-123.31045532+26.70539856%2C+-124.16611481+27.355196%2C+-124.7996521+28.01731682%2C+-125.1881485+28.62464333%2C+-125.60649109+29.39650726%2C+-126.06056976+30.08115196%2C+-126

In [62]:
## TODO try getting the bounding box geometry instead since the full region geom is HUGE